In [ ]:
import os, re, csv, json, time
from datasets import load_dataset
from groq import Groq
from openai import OpenAI
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


# Run Inference 

In [ ]:
API_KEY = "Your key here"

router_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
groq_client = groq_client = Groq(api_key=API_KEY)


In [ ]:

# ==========================================
# CONFIGURATION
# ==========================================
CSV_PATH = "data/MedCalc_Numeric_Results.csv"
QUESTIONS_JSON_PATH = "data/MedCalc_Numeric_Questions.json"
RAW_ANSWERS_JSON_PATH = "data/MedCalc_Numeric_Raw_Answers.json"



MODELS = [
    ("groq", "openai/gpt-oss-120b"),
    ("groq", "openai/gpt-oss-20b"),
    ("router", "poolside/laguna-s-2.1"),
    ("router", "microsoft/phi-4"),
    ("router", "qwen/qwen3-8b"),
    ("router", "mistralai/ministral-8b-2512"),
    ("router", "meta-llama/llama-4-scout"),
    ("router","nvidia/nemotron-3-nano-30b-a3b"),
    ("router","deepseek/deepseek-v3.2-exp")]


REASONING_MODELS = ("gpt-oss", "DeepSeek", "laguna", "nemotron", "qwen3", "qwen2.5")
MAX_TOKENS = 8192  

META_COLS = ["question_id", "calculator_name", "output_type", "correct_answer", "lower_limit", "upper_limit"]

# ==========================================
#  LOAD DATASET
# ==========================================
if os.path.exists(QUESTIONS_JSON_PATH):
    print(f"Loading cached numeric dataset from {QUESTIONS_JSON_PATH}...")
    with open(QUESTIONS_JSON_PATH, "r", encoding="utf-8") as f:
        saved_items = json.load(f)
else:
    print("Downloading dataset nsk7153/MedCalc-Bench-Verified...")
    raw_dataset = load_dataset("nsk7153/MedCalc-Bench-Verified", split="test")

    def is_numeric(example):
        return str(example.get("Output Type", "")).strip().lower() in ("decimal", "integer")

    filtered_dataset = raw_dataset.filter(is_numeric)
    print(f"Filtered {len(raw_dataset)} -> {len(filtered_dataset)} numeric questions.")

    saved_items = []
    for i, item in enumerate(filtered_dataset):
        saved_items.append({
            "question_id": f"NumQ{i+1}",
            "calculator_name": item.get("Calculator Name", "Unknown"),
            "output_type": str(item.get("Output Type", "")).strip().lower(),
            "patient_note": item.get("Patient Note", ""),
            "question": item.get("Question", ""),
            "answer": str(item.get("Ground Truth Answer", "")),
            "lower_limit": str(item.get("Lower Limit", "")),
            "upper_limit": str(item.get("Upper Limit", "")),
        })
    with open(QUESTIONS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(saved_items, f, indent=2)

print(f"Total numeric questions ready: {len(saved_items)}")

# ==========================================
# RESUME INFERENCE
# ==========================================
# existing_data[q][model]:  ""    -> never got a response (will retry)
#                           "N/A" -> got a response, no parseable answer (won't retry)
#                           "12.3"-> parsed prediction
existing_data = {item["question_id"]: {} for item in saved_items}
raw_text_data = {item["question_id"]: {} for item in saved_items}
all_known_models = set(m[1] for m in MODELS)

if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        model_cols = [c for c in (reader.fieldnames or []) if c not in META_COLS]
        all_known_models.update(model_cols)
        for row in reader:
            q_id = row["question_id"]
            if q_id in existing_data:
                for model in model_cols:
                    if row.get(model, "") != "":
                        existing_data[q_id][model] = row[model]

if os.path.exists(RAW_ANSWERS_JSON_PATH):
    with open(RAW_ANSWERS_JSON_PATH, "r", encoding="utf-8") as f:
        for entry in json.load(f):
            q_id = entry.get("question_id")
            if q_id in raw_text_data:
                raw_text_data[q_id] = entry.get("model_responses", {})

# ==========================================
#  PROMPTING & EXTRACTION
# ==========================================
def format_question(item):
    return (
        "You are an expert clinical calculator. Review the patient note and answer the question.\n"
        "First, think step by step to extract the parameters and perform the calculation. "
        "Then, you MUST end your response with the exact phrase: 'The answer is X' where X is your "
        "final calculated number (e.g., 15.2 or 45). Do not include units or any other text at the very end.\n\n"
        f"Patient Note:\n{item['patient_note']}\n\n"
        f"Question:\n{item['question']}\n\n"
        "Answer:"
    )

_NUM = r"[-+]?(?:\d*\.\d+|\d+)"
ANCHOR_RE = re.compile(r"answer is[^0-9+\-]{0,40}(" + _NUM + ")", re.IGNORECASE | re.DOTALL)

def extract_float(text):
    """Number after the LAST 'answer is'. None if no anchor -> counted wrong, not retried."""
    if not text:
        return None
    hits = ANCHOR_RE.findall(text.replace(",", ""))
    if not hits:
        return None
    try:
        return float(hits[-1])
    except ValueError:
        return None

def reextract_from_raw():
    """Recompute all predictions from stored raw text (use after changing extract_float)."""
    for q_id, responses in raw_text_data.items():
        for model, text in responses.items():
            all_known_models.add(model)
            if not text or text.strip() in ("", "N/A"):
                existing_data[q_id][model] = ""          # old failed call -> retry
                continue
            pred = extract_float(text)
            existing_data[q_id][model] = str(pred) if pred is not None else "N/A"

# ==========================================
#  INFERENCE API & SAVING
# ==========================================
def ask(provider, model, prompt, retries=4):
    client = groq_client if provider == "groq" else router_client
    kwargs = dict(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=MAX_TOKENS,
    )
    if any(tag in model for tag in REASONING_MODELS):
        kwargs["reasoning_effort"] = "medium"

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(**kwargs)
            choice = resp.choices[0]
            if choice.finish_reason == "length":
                print(f"  WARNING: {model} hit max_tokens")
            return choice.message.content
        except Exception as e:
            wait = 20 * (attempt + 1) if "429" in str(e) else 5 * (attempt + 1)
            print(f"  {model} error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(wait)
    return None

def save_csv():
    sorted_models = sorted(all_known_models)
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=META_COLS + sorted_models)
        writer.writeheader()
        for item in saved_items:
            q_id = item["question_id"]
            row = {
                "question_id": q_id,
                "calculator_name": item["calculator_name"],
                "output_type": item["output_type"],
                "correct_answer": item["answer"],
                "lower_limit": item["lower_limit"],
                "upper_limit": item["upper_limit"],
            }
            for model in sorted_models:
                row[model] = existing_data[q_id].get(model, "")
            writer.writerow(row)

def save_json():
    correct = {item["question_id"]: item["answer"] for item in saved_items}
    out = [
        {"question_id": q_id, "model_responses": responses, "correct_response": correct.get(q_id, "N/A")}
        for q_id, responses in raw_text_data.items()
    ]
    with open(RAW_ANSWERS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=4)

# ==========================================
#  SCORING 
# ==========================================
def is_correct(pred_str, lower, upper):
    try:
        return float(lower) <= float(pred_str) <= float(upper)
    except ValueError:
        return False

def summarize():
    print("\n=== Accuracy ===")
    for model in sorted(all_known_models):
        correct = answered = 0
        for item in saved_items:
            v = existing_data[item["question_id"]].get(model, "")
            if v == "":
                continue
            answered += 1
            correct += is_correct(v, item["lower_limit"], item["upper_limit"])
        if answered:
            pending = len(saved_items) - answered
            print(f"{model:45s} {correct:4d}/{answered:<4d} = {100*correct/answered:5.1f}%   ({pending} pending)")

# ==========================================
#  EXECUTION LOOP
# ==========================================
reextract_from_raw()   # applies the fixed extractor to any previous runs
save_csv()

print("\nStarting evaluation loop...")
for item in saved_items:
    q_id = item["question_id"]
    prompt = format_question(item)
    dirty = False

    for provider, model in MODELS:
        if existing_data[q_id].get(model, "") != "":
            continue

        print(f"Fetching {q_id} for {model}...")
        text = ask(provider, model, prompt)

        if not text or not text.strip():
            print("  -> no response, will retry next run")
            time.sleep(2.5)
            continue

        pred = extract_float(text)
        raw_text_data[q_id][model] = text
        existing_data[q_id][model] = str(pred) if pred is not None else "N/A"
        dirty = True

        ok = is_correct(existing_data[q_id][model], item["lower_limit"], item["upper_limit"])
        print(f"  -> Predicted: {pred} | Expected: {item['answer']} | {'OK' if ok else 'WRONG'}")
        time.sleep(2.5)

    if dirty:
        save_csv()
        save_json()

save_csv()
save_json()
summarize()
print("Evaluation complete.")